# ERA5-Land monthly climate of the Serrana mesoregion and of the State, 2012-2025

**FINN_SC · `gee/era5land_serrana_monthly.ipynb`** (co-authors' review of 2026-10-19, comment C12)

The interannual analysis of the manuscript (section 2.8) correlates the State totals with the State means of the
July-November fire-season climate. The co-authors asked for two checks: the climate of the Serrana, where most events
occur, and the July-September period, which holds 60.0% of the events of the State and 74.9% of those of the Serrana.

Same recipe as `gee/era5land_oni_fire_season.py` (authors' NotebookB): `ECMWF/ERA5_LAND/MONTHLY_AGGR`, bands
`total_precipitation_sum`, `temperature_2m`, `dewpoint_temperature_2m`, `volumetric_soil_water_layer_1`, polygon means
at 11132 m; precipitation in mm (× 1000), temperatures in °C, vapour-pressure deficit from the Tetens equation (clipped
at 0). Polygons: the State and the union of the 30 municipalities of the Serrana (IBGE 2024 mesh and the mesoregion
table of the repository).

Output: `data/external/era5land_SC_serrana_monthly_2012_2025.csv` (region, year, month, precip_mm, Tc, VPD_kPa,
soil_moist), then Spearman correlations (Holm within each response) for: State events vs Serrana climate
(July-November and July-September), State events vs State climate (July-September), and Serrana events vs Serrana
climate (both periods).

In [1]:
# 1. Packages (skip if already installed)
%pip install -q earthengine-api geemap geopandas pandas scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 29.4 MB/s eta 0:00:00


In [2]:
# 2. Parameters
from pathlib import Path
# repository root = parent of gee/ (local run); otherwise the current folder (e.g. Colab, with the input files uploaded)
REPO_ROOT = Path.cwd().resolve().parent
if not (REPO_ROOT / "config" / "config.yml").is_file():
    REPO_ROOT = None
PROJECT = "ee-jessicajcss"   # <-- Earth Engine Cloud project (change it to your own)
OUT_NAME = "era5land_SC_serrana_monthly_2012_2025.csv"
OUT = (REPO_ROOT / "data" / "external" / OUT_NAME) if REPO_ROOT else Path.cwd() / OUT_NAME
print("output:", OUT)

output: /content/era5land_SC_serrana_monthly_2012_2025.csv


In [3]:
# 3. Earth Engine authentication
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine initialised, project", PROJECT)

Earth Engine initialised, project ee-jessicajcss


In [4]:
# 4. Municipal polygons of Santa Catarina: IBGE 2024 mesh, the same used for the FINN and MapBiomas Fogo tables.
#    Earth Engine receives the simplified copy gee/SC_municipios_2024_simplified.gpkg (tools/simplify_municipios_for_gee.py:
#    shared borders simplified at 0.0005 degrees, about 55 m; area change per municipality <= 0.07 %), because the full
#    shapefile (about 70 MB as GeoJSON) exceeds the 10 MB limit of an Earth Engine request.
#    In Colab, this cell asks for that one file (folder gee/ of the repository, about 2.7 MB).
import geopandas as gpd, geemap, pandas as pd
GPKG = "SC_municipios_2024_simplified.gpkg"
if REPO_ROOT is not None:
    gpkg_path = REPO_ROOT / "gee" / GPKG
else:
    gpkg_path = Path.cwd() / GPKG
    if not gpkg_path.is_file():
        try:
            from google.colab import files
            print(f"Select the file {GPKG} (folder gee/ of the repository)")
            for name in files.upload():
                if name != GPKG and name.startswith("SC_municipios_2024_simplified"):
                    Path(name).rename(GPKG)          # Colab renames a repeated upload, e.g. "... (1).gpkg"
        except ImportError:
            pass
    assert gpkg_path.is_file(), f"copy {GPKG} (folder gee/ of the repository) to {Path.cwd()}"
mun_s = gpd.read_file(gpkg_path)
mun_s["CD_MUN"] = mun_s["CD_MUN"].astype("int64")
assert len(mun_s) == 295 and mun_s.CD_MUN.is_unique and mun_s.mesorregiao.notna().all(), len(mun_s)
if REPO_ROOT is not None:   # full-resolution mesh for the local steps
    mun = gpd.read_file(REPO_ROOT / "data" / "inputs" / "SC_Municipios_2024.shp", encoding="cp1252").to_crs(4326)
    mun["CD_MUN"] = mun["CD_MUN"].astype("int64")
    mun = mun.merge(mun_s[["CD_MUN", "mesorregiao"]], on="CD_MUN", how="left")
else:
    mun = mun_s
mun_ee = geemap.geopandas_to_ee(mun_s[["CD_MUN", "NM_MUN", "mesorregiao", "geometry"]])
print(len(mun_s), "municipalities;", mun_s.groupby("mesorregiao").size().to_dict())


Select the file SC_municipios_2024_simplified.gpkg (folder gee/ of the repository)


Saving SC_municipios_2024_simplified.gpkg to SC_municipios_2024_simplified.gpkg
295 municipalities; {'Grande Florianópolis': 21, 'Norte Catarinense': 26, 'Oeste Catarinense': 118, 'Serrana': 30, 'Sul Catarinense': 46, 'Vale do Itajaí': 54}


In [5]:
# 5. Monthly means over the State and over the Serrana
import numpy as np
regions = {"Santa Catarina": geemap.geopandas_to_ee(mun_s.dissolve()[["geometry"]]).geometry(),
           "Serrana": geemap.geopandas_to_ee(mun_s[mun_s.mesorregiao == "Serrana"].dissolve()[["geometry"]]).geometry()}
BANDS = ["total_precipitation_sum", "temperature_2m", "dewpoint_temperature_2m", "volumetric_soil_water_layer_1"]
ic = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").select(BANDS).filterDate("2012-01-01", "2026-01-01")
frames = []
for name, geom in regions.items():
    def _reduce(img, geom=geom):
        d = img.reduceRegion(ee.Reducer.mean(), geom, 11132, maxPixels=1e10)
        return ee.Feature(None, d).set("date", img.date().format("YYYY-MM-dd"))
    c = geemap.ee_to_df(ee.FeatureCollection(ic.map(_reduce)))
    c["region"] = name
    frames.append(c)
clim = pd.concat(frames, ignore_index=True)
clim["date"] = pd.to_datetime(clim["date"])
clim["year"], clim["month"] = clim.date.dt.year, clim.date.dt.month
clim["precip_mm"] = clim.total_precipitation_sum * 1000.0
clim["Tc"] = clim.temperature_2m - 273.15
Td = clim.dewpoint_temperature_2m - 273.15
es = 0.6108 * np.exp(17.27 * clim.Tc / (clim.Tc + 237.3)); ea = 0.6108 * np.exp(17.27 * Td / (Td + 237.3))
clim["VPD_kPa"] = (es - ea).clip(lower=0)
clim["soil_moist"] = clim.volumetric_soil_water_layer_1
clim = clim[["region", "year", "month", "precip_mm", "Tc", "VPD_kPa", "soil_moist"]].sort_values(["region", "year", "month"])
print(clim.groupby("region").size())

region
Santa Catarina    168
Serrana           168
dtype: int64


In [6]:
# 6. Save, and check the State July-November values against the archived file
import hashlib
OUT.parent.mkdir(parents=True, exist_ok=True)
clim.to_csv(OUT, index=False)
print("saved", OUT, "| SHA-256", hashlib.sha256(OUT.read_bytes()).hexdigest())
def season(df, months):
    s = df[df.month.isin(months)].groupby(["region", "year"])
    return s.agg(precip_mm=("precip_mm", "sum"), Tc=("Tc", "mean"), VPD_kPa=("VPD_kPa", "mean"), soil_moist=("soil_moist", "mean"))
JASON, JAS = season(clim, range(7, 12)), season(clim, [7, 8, 9])
if REPO_ROOT is not None:
    arch = pd.read_csv(REPO_ROOT / "data" / "external" / "era5land_oni_SC_fire_season_2012_2025.csv").set_index("year")
    new = JASON.loc["Santa Catarina"]
    print("max |difference| with the archived State July-November precipitation (mm):",
          round((new.precip_mm - arch.precip_mm_fs).abs().max(), 2), "(geometry: IBGE 2024 here, geobr 2022 in the archived run)")

saved /content/era5land_SC_serrana_monthly_2012_2025.csv | SHA-256 f459206e2eb413ffb87376fe1322d5f21a5fff02569e309b9381f5fca7d496e9


In [7]:
# 7. Spearman correlations (Holm within each response, 4 predictors)
from scipy import stats
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); out = np.empty(m); run = 0
    for k, i in enumerate(o):
        run = max(run, (m - k) * p[i]); out[i] = min(1.0, run)
    return out
if REPO_ROOT is None:
    raise SystemExit("run this cell inside the repository (it reads results/tables)")
E = pd.read_csv(REPO_ROOT / "results" / "tables" / "events.csv.gz")
E = E[(E.sensor == "MODIS+VIIRS") & (~E.flag) & E.year.between(2012, 2025)]
att = pd.read_csv(REPO_ROOT / "data" / "external" / "sc_municipios_2024_atributos.csv")
E["meso"] = E.mun_event.map(dict(zip(att.NM_MUN, att.mesorregiao)))
resp = {"State events": E.groupby("year").size(), "Serrana events": E[E.meso == "Serrana"].groupby("year").size()}
cases = [("State events", "Serrana", "Jul-Nov", JASON), ("State events", "Serrana", "Jul-Sep", JAS),
         ("State events", "Santa Catarina", "Jul-Sep", JAS), ("Serrana events", "Serrana", "Jul-Nov", JASON),
         ("Serrana events", "Serrana", "Jul-Sep", JAS)]
rows = []
for rname, region, period, tab in cases:
    X = tab.loc[region]; y = resp[rname].reindex(X.index)
    rr = [stats.spearmanr(y, X[v]) for v in ["precip_mm", "Tc", "VPD_kPa", "soil_moist"]]
    ph = holm([r[1] for r in rr])
    for v, r, p in zip(["precip_mm", "Tc", "VPD_kPa", "soil_moist"], rr, ph):
        rows.append(dict(response=rname, climate=region, period=period, predictor=v, n=len(y), rho=r[0], p=r[1], p_holm=p))
R = pd.DataFrame(rows)
print(R.round(3).to_string(index=False))

SystemExit: run this cell inside the repository (it reads results/tables)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Next steps

1. Copy the CSV to `data/external/` (if the notebook ran outside the repository) and record its SHA-256 in
   `docs/provenance.md`.
2. Fill the passage highlighted in yellow in section 2.8 of the manuscript and add the results to the Supplementary
   Material (section S15) with the values printed in cell 7.